# OmniVoice — Chuyển Văn Bản Thành Giọng Nói Tiếng Việt

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doanquangkien/OmniVoice/blob/master/TEST/Colab_OmniVoice_Vietnam.ipynb)

> **GPU Miễn Phí:** T4 16GB — nhanh hơn GPU local 4GB khoảng 10-20 lần.
> **Thời gian inference:** ~30-60 giây/audio.

---

## 📋 Hướng Dẫn Nhanh

1. Chạy từng cell theo thứ tự (Ctrl+Enter hoặc nút ▶)
2. Cell 1: Cài đặt (2-3 phút, chỉ chạy lần đầu)
3. Cell 2: Khởi động — mở link Gradio public
4. Dùng giao diện web tiếng Việt để tạo giọng nói

> ⚠️ **Lưu ý:** Colab Free có giới hạn ~4-6h GPU/ngày. Phiên sẽ tự ngắt sau ~30 phút không tương tác.

## 1. Cài Đặt OmniVoice (2-3 phút)

In [ ]:
# Bước 1: Cài OmniVoice từ GitHub (bản đã Việt hóa UI)
!pip install git+https://github.com/doanquangkien/OmniVoice.git --quiet

# Bước 2: Cài thêm text normalization (đọc số, ngày tháng chuẩn)
!pip install "omnivoice[tn]" --quiet

print("✅ Cài đặt hoàn tất!")

## 2. Khởi Động Giao Diện Web

Chạy cell này → đợi model load (~30 giây) → click link Gradio public để mở giao diện.

In [ ]:
!omnivoice-demo --share

---

## (Tùy Chọn) Python API — Voice Cloning Trực Tiếp

Nếu bạn muốn dùng code thay vì giao diện web:

In [ ]:
from omnivoice import OmniVoice
import soundfile as sf
from IPython.display import Audio, display
import torch

# Load model (T4 GPU)
model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16
)
print("✅ Model đã sẵn sàng!")

In [ ]:
# Upload audio tham chiếu (3-10 giây, giọng mẫu)
from google.colab import files

print("📁 Upload audio tham chiếu (file .wav / .mp3 / .flac):")
uploaded = files.upload()
ref_audio_path = list(uploaded.keys())[0]
print(f"✅ Đã upload: {ref_audio_path}")

In [ ]:
# Tạo giọng nói tiếng Việt từ audio tham chiếu
text = "Công nghệ trí tuệ nhân tạo đang thay đổi cách chúng ta sống và làm việc mỗi ngày."

print(f"🎵 Đang tạo giọng nói cho: \"{text}\"")
print("⏳ Đợi khoảng 30-60 giây...")

audio = model.generate(
    text=text,
    ref_audio=ref_audio_path,
    # ref_text="...",  # Bỏ dòng này nếu muốn tự động nhận dạng bằng Whisper
)

sf.write("output.wav", audio[0], 24000)
print("✅ Đã tạo xong! Nghe thử:")
display(Audio(audio[0], rate=24000))

---

## 📊 Benchmark Nhanh (So Sánh Tốc Độ)

Chạy cell này để đo thời gian inference trên Colab T4.

In [ ]:
import time

test_texts = [
    ("Ngắn (1 câu)", "Xin chào, tôi là trợ lý ảo OmniVoice."),
    ("Vừa (3 câu)", "Công nghệ trí tuệ nhân tạo đang thay đổi cách chúng ta sống và làm việc mỗi ngày. Từ những chiếc xe tự lái đến các trợ lý giọng nói thông minh, tất cả đều đang dần trở thành một phần không thể thiếu trong cuộc sống hiện đại."),
    ("Dài (đoạn văn)", "Việt Nam là một đất nước có bề dày lịch sử hơn bốn nghìn năm văn hiến. Từ những cánh đồng lúa bạt ngàn ở đồng bằng sông Cửu Long, đến những thửa ruộng bậc thang uốn lượn nơi vùng cao Tây Bắc, tất cả tạo nên một bức tranh thiên nhiên đa dạng và phong phú. Con người Việt Nam cần cù, thông minh và giàu lòng mến khách."),
]

print("📊 Benchmark OmniVoice trên Colab T4 (16GB VRAM):")
print("=" * 60)

for label, text in test_texts:
    print(f"\n🎵 {label}:")
    print(f"   Text: {text[:80]}..." if len(text) > 80 else f"   Text: {text}")
    
    t0 = time.time()
    audio = model.generate(
        text=text,
        ref_audio=ref_audio_path,
        num_step=32,
    )
    elapsed = time.time() - t0
    
    duration = len(audio[0]) / 24000
    rtf = elapsed / duration if duration > 0 else 0
    
    print(f"   ⏱️  Thời gian: {elapsed:.1f}s")
    print(f"   🎧 Audio dài: {duration:.1f}s")
    print(f"   ⚡ RTF: {rtf:.3f} (thấp hơn = nhanh hơn real-time)")

print("\n" + "=" * 60)
print("✅ Benchmark hoàn tất!")
print("💡 So sánh: GPU local 4GB mất 10-15 phút/audio với cùng text.")